<a href="https://colab.research.google.com/github/ndoteddy/mcp-web-performance-intelligence/blob/main/mcp_agent_performance_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enterprise Design: Agentic Model Context Protocol (MCP) Server

This technical specification and implementation guide establishes a robust agentic workflow utilizing **FastMCP** for interface definition, **Generative AI Tool Orchestration** for dynamic routing, and a **Persistent SQLite Data Layer** to maintain state across high-latency diagnostic tasks.

### System Architecture:
1. **Service Ingress:** Accepts URL-based performance analysis requests.
2. **Orchestration Layer:** Evaluates intent and maps requests to specific tool schemas via the FastMCP server.
3. **Analytical Engine (`analyze_website`):** Interfaces with the Google PageSpeed Insights API. It implements a verification logic to validate performance metrics before persisting raw data and executive summaries to the SQLite storage layer.
4. **Inference Engine (`suggest_improvements`):** Retrieves historical metrics from the persistence layer to generate prioritized technical remediation strategies without redundant API calls.

In [1]:
# Step 1: Install Dependencies
!pip install -q fastmcp httpx google-generativeai

### Environment Configuration
Define required API credentials within the execution environment.

In [2]:
import os
import sqlite3
import json
import httpx
import google.generativeai as genai
from google.colab import userdata
from fastmcp import FastMCP

# ==========================================
# CONFIGURATION & API KEYS (Using Secrets)
# ==========================================

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    PAGESPEED_API_KEY = userdata.get('PAGESPEED_API_KEY')

    if not GOOGLE_API_KEY:
        print("⚠️ GOOGLE_API_KEY not found in secrets!")
    else:
        os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
        genai.configure(api_key=GOOGLE_API_KEY)
        # Using standard flash-latest alias
        model = genai.GenerativeModel('gemini-flash-latest')
        print("✅ Gemini API configured.")

    if not PAGESPEED_API_KEY:
        print("⚠️ PAGESPEED_API_KEY not found in secrets!")

except Exception as e:
    print(f"❌ Error loading secrets: {e}")

# Initialize SQLite Local Memory DB
DB_FILE = "mcp_performance.db"

def init_db():
    """Creates the performance cache table if it doesn't exist."""
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS performance_cache (
                url TEXT PRIMARY KEY,
                metrics TEXT,
                analysis_summary TEXT,
                updated_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.commit()

init_db()
print("💾 SQLite Local Memory Database initialized successfully!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini API configured.
💾 SQLite Local Memory Database initialized successfully!


### Tool Definition and FastMCP Server Integration
The following section initializes the FastMCP server and registers the core diagnostic tools for website performance evaluation.

In [60]:
import time

mcp = FastMCP("WebPerformanceAnalyzer")

def fetch_pagespeed_data(url: str) -> dict:
    # Increased timeout to 60.0s for high-latency diagnostics
    api_url = f"https://www.googleapis.com/pagespeedonline/v5/runPagespeed?url={url}&key={PAGESPEED_API_KEY}"
    response = httpx.get(api_url, timeout=60.0)
    response.raise_for_status()
    data = response.json()

    # Printing raw response for internal transparency
    print(f"\n[DEBUG] Raw PageSpeed API Response for {url}:\n{json.dumps(data, indent=2)[:500]}... (truncated)")

    res = data.get("lighthouseResult", {})
    audits = res.get("audits", {})

    metrics = {
        "score": res.get("categories", {}).get("performance", {}).get("score", 0) * 100,
        "first_contentful_paint": audits.get("first-contentful-paint", {}).get("displayValue", "N/A"),
        "speed_index": audits.get("speed-index", {}).get("displayValue", "N/A"),
        "interactive": audits.get("interactive", {}).get("displayValue", "N/A"),
        "total_blocking_time": audits.get("total-blocking-time", {}).get("displayValue", "N/A")
    }
    return metrics

@mcp.tool()
def analyze_website(url: str) -> str:
    """Analyzes website performance and returns raw JSON metrics and technical breakdown."""
    print(f"\n[MCP Tool] Calling PageSpeed for: {url}")
    try:
        metrics = fetch_pagespeed_data(url)
        raw_json = json.dumps(metrics, indent=2)
        return f"RAW_METRICS:\n{raw_json}"
    except Exception as e:
        return f"Error fetching data: {str(e)}"

print("⚙️ Tools updated with extended timeout (60s).")

⚙️ Tools updated with extended timeout (60s).


In [39]:
import httpx
import json

# Diagnostic Cell: Test PageSpeed API directly without the Agent
def test_api_connection(url="https://google.com"):
    print(f"Testing PageSpeed API for {url}...")
    api_url = f"https://www.googleapis.com/pagespeedonline/v5/runPagespeed?url={url}&key={PAGESPEED_API_KEY}"
    try:
        response = httpx.get(api_url, timeout=30.0)
        if response.status_code == 200:
            print("✅ API Connection Successful!")
            data = response.json()
            score = data.get('lighthouseResult', {}).get('categories', {}).get('performance', {}).get('score')
            print(f"Performance Score: {score}")
        else:
            print(f"❌ API Error {response.status_code}: {response.text}")
    except Exception as e:
        print(f"❌ Connection Failed: {e}")

test_api_connection()

Testing PageSpeed API for https://google.com...
✅ API Connection Successful!
Performance Score: 0.86


### Performance Tool Schema Optimization
Ensuring strict string serialization for the analytical response to maintain compatibility with model-driven function calling.

In [53]:
@mcp.tool()
def analyze_website(url: str) -> str:
    """Analyzes website performance and returns raw JSON metrics. Required for technical analysis."""
    try:
        metrics = fetch_pagespeed_data(url)
        metrics_json = json.dumps(metrics)

        # Persist to SQLite so 'suggest_improvements' can access it
        with sqlite3.connect(DB_FILE) as conn:
            cursor = conn.cursor()
            cursor.execute(
                "INSERT OR REPLACE INTO performance_cache (url, metrics) VALUES (?, ?)",
                (url, metrics_json)
            )
            conn.commit()

        return metrics_json
    except Exception as e:
        return f"ERROR: {str(e)}"

[05/16/26 07:58:30] WARNING  Component already exists: tool:analyze_website@                  ]8;id=808397;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=189301;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

### Intent Routing and Tool Orchestration Logic
Definition of the function calling schema used by the orchestrator to identify and sequence multi-step operations.

In [17]:
mcp_tools_gemini = [
    {
        "function_declarations": [
            {
                "name": "analyze_website",
                "description": "Analyzes website performance. Returns speed status and metrics. Use this first if no analysis exists.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {"type": "string", "description": "The full URL of the website to analyze"}
                    },
                    "required": ["url"]
                }
            },
            {
                "name": "suggest_improvements",
                "description": "Provides 3 actionable technical recommendations based on existing analysis in the database.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {"type": "string", "description": "The full URL of the website to get suggestions for"}
                    },
                    "required": ["url"]
                }
            }
        ]
    }
]

print("⠖ Agent routing matrix updated for multi-step reasoning.")

⠖ Agent routing matrix updated for multi-step reasoning.


### Diagnostic Execution Interface
Execution harness for the performance audit workflow, including quota management and sequential task execution.

In [47]:
import google.generativeai as genai
import time
import json

print("=======================================================")
print("☇ MCP Agent Performance Detailed Terminal ☇")
print("=======================================================\n")

def call_with_retry(func, *args, **kwargs):
    for i in range(3):
        try:
            res = func(*args, **kwargs)
            if res: return res
        except Exception as e:
            if "429" in str(e):
                wait_time = (i + 1) * 40
                print(f"\n⏳ Quota hit (429). Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise e
    return None

try:
    tools_list = [analyze_website, suggest_improvements]
    model_name = 'gemini-2.5-flash-lite'
    model_with_tools = genai.GenerativeModel(model_name=model_name, tools=tools_list)
    chat = model_with_tools.start_chat(enable_automatic_function_calling=True)

    user_input = "Analyze the performance of https://google.com using your tools. Provide the raw JSON metrics followed by a senior-level technical analysis."
    print(f"⡤ You: {user_input}")

    print("⏳ Initial cooling down to preserve quota...")
    time.sleep(15)

    # 1. Execute analysis turn
    response = call_with_retry(chat.send_message, user_input)

    if response:
        print("\n[DEBUG] Tool Execution Trace:")
        for message in chat.history:
            if hasattr(message, 'parts'):
                for part in message.parts:
                    if hasattr(part, 'function_response') and part.function_response:
                        try:
                            res_dict = dict(part.function_response.response)
                            print(f" └ ✅ Captured Tool Data: {json.dumps(res_dict)[:150]}...")
                        except:
                            print(" └ ⚠️ Could not serialize tool part")

        # 2. Final Synthesis Request
        print("\n⌛ Generating Senior Technical Report...")
        time.sleep(10)
        final_prompt = "Excellent. Now, as a Senior Performance Engineer, provide the full technical report and include the raw JSON metrics in a code block."
        final_response = call_with_retry(chat.send_message, final_prompt)

        if final_response and hasattr(final_response, 'text'):
            print(f"\n✅ FINAL DETAILED REPORT:\n{final_response.text}")
        else:
            print("\n☐ Model failed to generate final text.")
    else:
        print("\n❌ Failed to get initial response from model after retries.")

except Exception as e:
    print(f"\n⡂ Terminal Error: {e}")

☇ MCP Agent Performance Detailed Terminal ☇

⡤ You: Analyze the performance of https://google.com using your tools. Provide the raw JSON metrics followed by a senior-level technical analysis.
⏳ Initial cooling down to preserve quota...

[DEBUG] Raw PageSpeed API Response for https://google.com:
{
  "captchaResult": "CAPTCHA_NOT_NEEDED",
  "kind": "pagespeedonline#result",
  "id": "https://www.google.com/sorry/index?continue=https://www.google.com/&q=EgRC-VHkGLO-oNAGIileU4SiZUFqJhrOmBSSSySwaPTSLuJMXJzqHiTGZRNOcX02t3JyeFEmsTICclJaAUM",
  "loadingExperience": {
    "id": "https://www.google.com/sorry/index",
    "metrics": {
      "EXPERIMENTAL_TIME_TO_FIRST_BYTE": {
        "percentile": 1164,
        "distributions": [
          {
            "min": 0,
            "max": 800,
            "proportion": 0.6512
          },
          {
            "min": 800,
            "max": 1800,
            "proportion": 0.2123
          },
          {
            "min": 1800,
            "proportion"

### Refactored Class-Based Architecture

Structural refactoring following modern software engineering principles:
1. **Encapsulation**: Centralized logic within the `PerformanceAgent` class.
2. **Modularity**: Separation of prompt templates, configuration parameters, and execution logic.
3. **Extensibility**: Design pattern supports the seamless integration of additional diagnostic tools.

In [61]:
import os
import json
import time
import logging
import sqlite3
from typing import List, Optional, Any
import google.generativeai as genai

# --- Configuration Parameters ---
CONFIG = {
    "model_name": "gemini-2.5-flash-lite",
    "target_url": "https://google.com",
}

PROMPTS = {
    "system_instruction": (
        "You are a Senior Performance Engineer. Use your tools to perform a two-step audit: "
        "1. Analyze the website speed using 'analyze_website'. "
        "2. Immediately follow up with 'suggest_improvements' for that same URL. "
        "Finally, provide a high-detail technical report including the raw JSON metrics in a code block."
    ),
    "analysis_request": "Please conduct a full performance audit and improvement plan for: {url}"
}

# --- Logging Specification ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class PerformanceAgent:
    """Architecture for automated Website Performance Analysis."""
    def __init__(self, model_name: str, tools: List[Any]):
        self.model = genai.GenerativeModel(
            model_name=model_name,
            tools=tools,
            system_instruction=PROMPTS["system_instruction"]
        )
        self.chat = self.model.start_chat(enable_automatic_function_calling=True)

    def run_audit(self, url: str):
        """Executes the end-to-end performance audit sequence with retry logic."""
        logger.info(f"Initializing performance audit: {url}")

        for attempt in range(2):
            try:
                prompt = PROMPTS["analysis_request"].format(url=url)
                response = self.chat.send_message(prompt)

                print("\n" + "="*70)
                print("TECHNICAL PERFORMANCE AUDIT REPORT")
                print("="*70 + "\n")
                print(response.text)

                return response
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {str(e)}")
                if attempt == 0:
                    time.sleep(5)
                    continue
                logger.error(f"Audit execution failure: {str(e)}")
                return None

# --- Main Entry Point ---
if __name__ == "__main__":
    # Ensure tools are initialized correctly
    audit_agent = PerformanceAgent(
        model_name=CONFIG["model_name"],
        tools=[analyze_website, suggest_improvements]
    )

    audit_agent.run_audit(url=CONFIG["target_url"])


[MCP Tool] Calling PageSpeed for: https://google.com

[DEBUG] Raw PageSpeed API Response for https://google.com:
{
  "captchaResult": "CAPTCHA_NOT_NEEDED",
  "kind": "pagespeedonline#result",
  "id": "https://www.google.com/sorry/index?continue=https://www.google.com/&q=EgRA6ayNGNnLoNAGIik1z73C-jbQSVyxyrOSbqX1IPOs7fZjiZEfcj6jHp1TFihX3DIf9r1VgzICclJaAUM",
  "loadingExperience": {
    "id": "https://www.google.com/sorry/index",
    "metrics": {
      "EXPERIMENTAL_TIME_TO_FIRST_BYTE": {
        "percentile": 1164,
        "distributions": [
          {
            "min": 0,
            "max": 800,
          ... (truncated)

TECHNICAL PERFORMANCE AUDIT REPORT

As a Senior Performance Engineer, I've completed a two-step audit of https://google.com.

**Initial Analysis:**

The website exhibits a "Fast Paint, Slow Response" performance profile.

*   **First Contentful Paint (FCP):** 0.5 seconds
*   **Speed Index:** 0.8 seconds
*   **Time to Interactive (TTI):** 2.5 seconds
*   **Total Block

In [58]:
readme_content = """# 🚀 MCP Performance Agent

An agentic workflow utilizing the **Model Context Protocol (MCP)** to analyze website performance via the Google PageSpeed API and Gemini models.

## ✨ Features
- **Automated Tool Discovery**: Uses FastMCP for clean tool definition.
- **Two-Step Agentic Flow**: Automatically sequences analysis and suggestion tools via Gemini's function calling.
- **Production Architecture**: Decoupled prompt management and class-based structure.
- **Persistence**: Local SQLite cache stores metrics for fast retrieval and state management.

## 🛠️ Setup
1. Store your `GOOGLE_API_KEY` and `PAGESPEED_API_KEY` in your environment or secrets manager.
2. Install dependencies: `pip install fastmcp google-generativeai httpx`.
3. Run the `PerformanceAgent` through the main entry point.

## 📋 Architecture
1. **Agent (Gemini)**: Acts as the brain, determining which tool to call.
2. **MCP Server**: Exposes `analyze_website` and `suggest_improvements`.
3. **Data Layer**: Google PageSpeed Insights V5.
4. **Memory Layer**: SQLite (`mcp_performance.db`) for caching raw JSON data.

## 🤝 Contributing
Contributions are welcome! Please feel free to submit a Pull Request.
"""

with open("README.md", "w") as f:
    f.write(readme_content)

print("✅ Clean README.md generated (Sample Output removed)!")

✅ Clean README.md generated (Sample Output removed)!
